In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Your files will be available at:
# /content/drive/MyDrive/your_file

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Convert Excel to CSV (Run this cell)
import pandas as pd

# Read Excel files
ratings_df = pd.read_excel("/content/drive/MyDrive/Lstm_project /ratings.xlsx")
movies_df = pd.read_excel("/content/drive/MyDrive/Lstm_project /movies.xlsx")

# Save as CSV
ratings_df.to_csv("/content/drive/MyDrive/Lstm_project /ratings.csv", index=False)
movies_df.to_csv("/content/drive/MyDrive/Lstm_project /movies.csv", index=False)

print("✅ Converted to CSV successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Converted to CSV successfully!


In [14]:
# STEP 2: Install/Import Libraries (Run this cell)
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [16]:
# STEP 3: Load Your Data (Run this cell)
# Change the folder name if you used something else
ratings_path = "/content/drive/MyDrive/Lstm_project /ratings.csv"
movies_path = "/content/drive/MyDrive/Lstm_project /movies.csv"

df = pd.read_csv(ratings_path)
movies_df = pd.read_csv(movies_path, encoding='latin1')
print("✅ Data loaded successfully!")

✅ Data loaded successfully!


In [17]:
# STEP 4: Run the original code (Run this cell)
# Copy your entire original code here
# Make sure to REPLACE the file paths with the ones above

# Mapping: movieId -> title and genres
movieId_to_title = dict(zip(movies_df['movieId'], movies_df['title']))
movieId_to_genres = dict(zip(movies_df['movieId'], movies_df['genres']))

# Sort data by user and timestamp
df = df.sort_values(by=['userId', 'timestamp'])

# Create sequences of movie interactions for each user
user_sequences = df.groupby('userId')['movieId'].apply(list)

# Convert movie IDs to a contiguous range
unique_movies = sorted(df['movieId'].unique())
movie_to_index = {movie: i + 1 for i, movie in enumerate(unique_movies)}
index_to_movie = {i + 1: movie for i, movie in enumerate(unique_movies)}

# Map index to movie title and genres
index_to_movie_title = {index: movieId_to_title.get(movie_id, "Unknown Title") for index, movie_id in index_to_movie.items()}
index_to_movie_genres = {index: movieId_to_genres.get(movie_id, "Unknown Genre") for index, movie_id in index_to_movie.items()}

# Map movieId in df to indexes
df['movieId'] = df['movieId'].map(movie_to_index)

# Prepare input-output sequences
sequence_length = 5
X, y = [], []

for user_movies in user_sequences:
    indexed_movies = [movie_to_index[m] for m in user_movies if m in movie_to_index]
    for i in range(len(indexed_movies) - sequence_length):
        X.append(indexed_movies[i:i + sequence_length])
        y.append(indexed_movies[i + sequence_length])

X = pad_sequences(X, maxlen=sequence_length, padding='pre')
y = np.array(y)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build model
vocab_size = len(movie_to_index) + 1
embedding_dim = 50

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim),
    LSTM(64),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train model
history = model.fit(X_train, y_train, epochs=2, batch_size=8, validation_data=(X_test, y_test))

# Model summary
model.summary()

# Recommendation function
def recommend_next_movie(movie_sequence):
    movie_sequence_idx = [movie_to_index.get(m, 0) for m in movie_sequence]
    movie_sequence_idx = pad_sequences([movie_sequence_idx], maxlen=sequence_length, padding='pre')
    prediction = model.predict(movie_sequence_idx, verbose=0)
    predicted_index = np.argmax(prediction, axis=-1)[0]

    movie_id = index_to_movie.get(predicted_index, "Unknown ID")
    movie_title = index_to_movie_title.get(predicted_index, "Unknown Title")
    movie_genre = index_to_movie_genres.get(predicted_index, "Unknown Genre")

    return movie_id, movie_title, movie_genre

# Example usage
example_sequence = user_sequences.iloc[0][:sequence_length]
predicted_movie_id, predicted_movie_title, predicted_movie_genre = recommend_next_movie(example_sequence)

print("\n✅ Prediction Results:")
print("Predicted next movie ID:", predicted_movie_id)
print("Predicted next movie title:", predicted_movie_title)
print("Predicted next movie genre:", predicted_movie_genre)

Epoch 1/2
9779/9779 ━━━━━━━━━━━━━━━━━━━━ 224s 22ms/step - accuracy: 0.0023 - loss: 8.1726 - val_accuracy: 0.0039 - val_loss: 7.9742
Epoch 2/2
9779/9779 ━━━━━━━━━━━━━━━━━━━━ 216s 22ms/step - accuracy: 0.0056 - loss: 7.6470 - val_accuracy: 0.0069 - val_loss: 7.8491


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 5, 50)          │       486,250 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        29,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 9725)           │       632,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,443,447 (13.14 MB)

 Trainable params: 1,147,815 (4.38 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,295,632 (8.76 MB)


✅ Prediction Results:
Predicted next movie ID: 1580
Predicted next movie title: Men in Black (a.k.a. MIB) (1997)
Predicted next movie genre: Action|Comedy|Sci-Fi
